In [2]:
import os
import geopandas as gpd
import pandas as pd

import gc

In [4]:
path = '../../adatok/fix/overpass_download'

### Pontok

In [13]:
# beolvasom a több geojosn-fájlt mindegyiket egy külön df-be
pontok_df = [
    gpd.read_file(os.path.join(path, 'pontok', f))
    for f in os.listdir(os.path.join(path, 'pontok'))
    if f.endswith('.geojson')
]

# 1. egyesitem a sok kis df-et
# 2. overpass letöltési hibákból származó duplikált elemek törtéle (csak az első elem marad meg)
pontok = (
    gpd.GeoDataFrame(pd.concat(pontok_df, ignore_index=True), crs=pontok_df[0].crs)
    .drop_duplicates(subset='id')
    .reset_index(drop=True)
)

# memória felszabítás (kissebb df-ek törlése memóriából)
del pontok_df
gc.collect()

Skipping field happy_hours:food: unsupported OGR type: 10


0

In [14]:
len(pontok)

190105

In [15]:
pontok.head()

,id,@id,abandoned:amenity,abandoned:shop,access,addr:city,addr:conscriptionnumber,addr:door,addr:flats,addr:floor,...,staff,stop,toys:rc,url:B612,url:borzsony,url:dabas,url:opten,was:entrance,web,yacht_rental
0,node/16947670,node/16947670,None,None,None,Orfű,None,None,None,None,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,node/25546152,node/25546152,None,None,None,Pécs,None,None,None,None,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,node/33123162,node/33123162,None,None,None,Fonyód,None,None,None,None,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,node/33127897,node/33127897,None,None,None,Tab,None,None,None,None,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,node/94743595,node/94743595,None,None,None,Somogyszob,None,None,None,None,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [16]:
megtartott_oszlopok = [
    'id', 
    'geometry',
    'addr:city', 
    'addr:postcode', 
    'addr:street', 
    'addr:housenumber',
]
pontok = pontok[megtartott_oszlopok]
pontok = pontok[pontok['addr:housenumber'].notna()].reset_index(drop=True)

In [17]:
len(pontok)

190105

In [18]:
pontok.head()

,id,geometry,addr:city,addr:postcode,addr:street,addr:housenumber
0,node/16947670,POINT (18.13766 46.14637),Orfű,7677,Dollár út,1
1,node/25546152,POINT (18.22533 46.06637),Pécs,7623,Indóház tér,1
2,node/33123162,POINT (17.55781 46.7501),Fonyód,8640,Ady Endre utca,2
3,node/33127897,POINT (18.04147 46.7341),Tab,8660,Kossuth Lajos utca,1
4,node/94743595,POINT (17.30675 46.28901),Somogyszob,7563,Keleti köz,1


### Épületek

In [5]:
megtartott_oszlopok = [
    'id', 
    'geometry',
    'addr:city', 
    'addr:postcode', 
    'addr:street', 
    'addr:housenumber',
    'addr:place',                # utca helyett kis falvakban
    'addr:conscriptionnumber',   # HRSZ
]

epuletek_df = []
for f in os.listdir(os.path.join(path, 'epuletek')):
    if not f.endswith('.geojson'):
        continue
    
    gdf = gpd.read_file(os.path.join(path, 'epuletek', f))
    
    # sorok szűrése: legyen housenumber VAGY place VAGY conscriptionnumber
    gdf = gdf[
        gdf['addr:housenumber'].notna() 
        | gdf['addr:place'].notna() 
        | gdf['addr:conscriptionnumber'].notna()
    ]
    
    # oszlopok szűkítése
    gdf = gdf[[c for c in megtartott_oszlopok if c in gdf.columns]]
    
    # poligon -> centroid (EOV-ban vetítve a pontosság miatt, vissza WGS84-be)
    eredeti_crs = gdf.crs
    gdf['geometry'] = gdf.geometry.to_crs(epsg=23700).centroid.to_crs(eredeti_crs)
    
    epuletek_df.append(gdf)

# egyesítés és duplikáltak törlése
epuletek = (
    gpd.GeoDataFrame(pd.concat(epuletek_df, ignore_index=True), crs=epuletek_df[0].crs)
    .drop_duplicates(subset='id')
    .reset_index(drop=True)
)

del epuletek_df
gc.collect()

0

In [6]:
len(epuletek)

468991

In [7]:
epuletek.head()

,id,geometry,addr:city,addr:postcode,addr:street,addr:housenumber,addr:place,addr:conscriptionnumber
0,relation/2212122,POINT (17.90868 47.09106),Veszprém,8200,Megyeház tér,5,None,None
1,relation/2533451,POINT (17.91415 47.08942),Veszprém,8200,Bajcsy-Zsilinszky Endre utca,2,None,None
2,relation/2533452,POINT (17.90626 47.09398),Veszprém,8200,Óváros tér,3,None,None
3,relation/2580968,POINT (17.91744 47.08836),Veszprém,8200,Radnóti Miklós tér,2,None,None
4,relation/2581249,POINT (17.91265 47.09168),Veszprém,8200,Mindszenty József utca,3-5,None,None


In [9]:
print("Csak place (housenumber és conscriptionnumber nélkül):",
      ((epuletek['addr:housenumber'].isna()) 
       & (epuletek['addr:place'].notna()) 
       & (epuletek['addr:conscriptionnumber'].isna())).sum())

print("Csak conscriptionnumber (housenumber és place nélkül):",
      ((epuletek['addr:housenumber'].isna()) 
       & (epuletek['addr:place'].isna()) 
       & (epuletek['addr:conscriptionnumber'].notna())).sum())

print("Place ÉS conscriptionnumber (housenumber nélkül):",
      ((epuletek['addr:housenumber'].isna()) 
       & (epuletek['addr:place'].notna()) 
       & (epuletek['addr:conscriptionnumber'].notna())).sum())

Csak place (housenumber és conscriptionnumber nélkül): 43
Csak conscriptionnumber (housenumber és place nélkül): 1114
Place ÉS conscriptionnumber (housenumber nélkül): 31


In [10]:
# csak housenumber marad, place és conscriptionnumber oszlop törlése
epuletek = epuletek.drop(columns=['addr:place', 'addr:conscriptionnumber'])

# sorok szűrése: csak housenumber-esek maradnak
epuletek = epuletek[epuletek['addr:housenumber'].notna()].reset_index(drop=True)

In [11]:
len(epuletek)

467803

In [12]:
epuletek.head()

,id,geometry,addr:city,addr:postcode,addr:street,addr:housenumber
0,relation/2212122,POINT (17.90868 47.09106),Veszprém,8200,Megyeház tér,5
1,relation/2533451,POINT (17.91415 47.08942),Veszprém,8200,Bajcsy-Zsilinszky Endre utca,2
2,relation/2533452,POINT (17.90626 47.09398),Veszprém,8200,Óváros tér,3
3,relation/2580968,POINT (17.91744 47.08836),Veszprém,8200,Radnóti Miklós tér,2
4,relation/2581249,POINT (17.91265 47.09168),Veszprém,8200,Mindszenty József utca,3-5


---

### Egyesítés

In [21]:
osm = gpd.GeoDataFrame(
    pd.concat([pontok, epuletek], ignore_index=True),
    crs=pontok.crs
)

# törlés memóriából
del pontok, epuletek
gc.collect()

0

In [22]:
len(osm)

657908

In [23]:
# duplikátum-szűrés postcode + street + housenumber alapján
# (ha ezek megegyeznek, ugyanaz a cím — akár pontként, akár épületként szerepelt az OSM-ben)
osm = osm.drop_duplicates(
    subset=['addr:postcode', 'addr:street', 'addr:housenumber'],
    keep='first'
).reset_index(drop=True)

In [25]:
print(f"Egyesítés után: {len(osm)} sor")

Egyesítés után: 623656 sor


In [26]:
osm.to_parquet('../../adatok/fix/osm_cimek_teljes.parquet')